<a href="https://colab.research.google.com/github/WallaceKingston/nintendo.creativity.design/blob/main/Solioquiffy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch

To load a pre-trained model, you typically import the `torch` library and `torchvision.models`. `torchvision.models` provides access to many popular models pre-trained on large datasets like ImageNet.

In [ ]:
import torch
import torchvision.models as models
from torchvision.models import ResNet18_Weights

# Load a pre-trained ResNet18 model
# Use 'weights=ResNet18_Weights.DEFAULT' to get the most up-to-date weights
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

print("Pre-trained ResNet18 model loaded successfully!")

# You can inspect the model architecture
# print(model)

# To use the model for inference, set it to evaluation mode
model.eval()

print("Model set to evaluation mode.")

Pre-trained ResNet18 model loaded successfully!
Model set to evaluation mode.


First, we need a sample image. We'll download one and then define the transformations required for a ResNet model pre-trained on ImageNet.

In [ ]:
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

# Define the transformations
# These are standard for models pre-trained on ImageNet
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Download a sample image (e.g., a cat image)
img_url = 'https://hips.hearstapps.com/hmg-prod/images/cute-cat-photos-1593441022.jpg?crop=0.668xw:1.00xh;0.166xw,0&resize=640:*'
response = requests.get(img_url)
img = Image.open(BytesIO(response.content))

print("Sample image downloaded and loaded.")

# Preprocess the image
input_tensor = transform(img)

# Add a batch dimension
input_batch = input_tensor.unsqueeze(0)  # create a mini-batch as expected by the model

print("Image preprocessed and ready for inference.")

Sample image downloaded and loaded.
Image preprocessed and ready for inference.


Now, we can perform inference by passing the preprocessed image through the model. The output will be raw scores (logits) for each class, which we'll convert to probabilities.

In [ ]:
import torch.nn.functional as F

# Move the input to the same device as the model (e.g., CPU or GPU)
# For this example, assuming model is on CPU
if torch.cuda.is_available():
    input_batch = input_batch.to('cuda')
    model.to('cuda')

with torch.no_grad(): # Disable gradient calculation for inference
    output = model(input_batch)

# Apply softmax to get probabilities
probabilities = F.softmax(output, dim=1)

# Get the top 5 predicted classes
top5_prob, top5_catid = torch.topk(probabilities, 5)

print("Inference complete. Top 5 probabilities:", top5_prob)
print("Top 5 class IDs:", top5_catid)

Inference complete. Top 5 probabilities: tensor([[0.6462, 0.1487, 0.0995, 0.0739, 0.0189]])
Top 5 class IDs: tensor([[283, 285, 282, 281, 287]])


To understand what these class IDs mean, we need the ImageNet class labels. You can typically find these in a separate file or directly from `torchvision.models` if it provides such a mapping. For now, I'll provide a placeholder for loading these labels and demonstrating how to map the predictions.

In [ ]:
# In a real scenario, you would load ImageNet class labels.
# For example, from a JSON file or a list.
# Here's a dummy example:

# You can often find a mapping from here:
# https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt

# Dummy class names for demonstration purposes
# In practice, you would load a list of 1000 ImageNet class labels.
class_names = [
    "tench", "goldfish", "great white shark", "tiger shark", "hammerhead",
    "electric ray", "stingray", "cock", "hen", "ostrich", "" # ... and 990 more
]
# A more robust way to get ImageNet class names:
# !wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
# with open("imagenet_classes.txt", "r") as f:
#     categories = [s.strip() for s in f.readlines()]

# Let's try to fetch actual ImageNet classes for better illustration
try:
    # This URL is a common source for ImageNet classes
    labels_url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
    response = requests.get(labels_url)
    imagenet_classes = [s.strip() for s in response.text.split('\n') if s.strip()]

    print("\nPredicted classes and probabilities:")
    for i in range(top5_prob.size(0)):
        print(f"  {imagenet_classes[top5_catid[0, i]]}: {top5_prob[0, i].item():.4f}")
except Exception as e:
    print(f"Could not fetch ImageNet class labels: {e}")
    print("Using dummy class names for demonstration (if available).")
    if len(class_names) > top5_catid[0, 0].item():
        for i in range(top5_prob.size(0)):
            print(f"  Class ID {top5_catid[0, i].item()}: {top5_prob[0, i].item():.4f}")
    else:
        print("No labels available to map predictions.")


Predicted classes and probabilities:
  Persian cat: 0.6462


To save the pre-trained model for later use, you can save its state dictionary. This allows you to reload the weights into the same model architecture without having to re-download the entire pre-trained model every time.

In [ ]:
import torch

# Define a path to save the model state dictionary
model_save_path = 'resnet18_pretrained_weights.pth'

# Save the model's state dictionary
torch.save(model.state_dict(), model_save_path)

print(f"Model state dictionary saved to {model_save_path}")